# Data loading and exploration, Data preprocessing and preparation for analysis

## Exploratory data analysis in Python and hypothesis testing.

- Author: Yauheni Khatsuk
- Date: 20.11.2025

## Project Goals and Objectives

*The goal of the project is to analyze the Yandex Afisha service, first to check the data and calculate several metrics in SQL, create a dashboard in Datalens. Then extract, preprocess, and check the obtained data in Python.  
Based on the results of the data analysis, we will prepare an overall conclusion and recommendations, which will describe:  
Conclusions on the preprocessing of the data we investigated.  
Analysis of the distribution of orders by segments and their seasonal changes, including autumn user activity, also examining popular events and partners.  
Statistical analysis of the data, testing several hypotheses regarding the activity of users on mobile and stationary devices.*   
*Key results of the analysis: which events are most in demand, how the popularity of events changed in the fall, how the average check amount changed. What information can be obtained after studying user activity? Are there clear leaders among regions and partners in terms of the number of orders and ticket sales revenue? Supplement the conclusion with other information that seems important and interesting to you. Keep the overall volume of conclusions compact and concise.*

## Data Description
*The first dataset 'final_tickets_orders_df.csv' includes information about all ticket orders made from two types of devices: mobile and desktop:
order_id — unique order identifier.
user_id — unique user identifier.
created_dt_msk — order creation date (Moscow time).
created_ts_msk — order creation date and time (Moscow time).
event_id — event identifier from the events table.
cinema_circuit — cinema chain. If not applicable, the value will be 'none'.
age_limit — age restriction of the event.
currency_code — payment currency, for example rub for Russian rubles.
device_type_canonical — type of device from which the order was made, for example mobile for mobile devices, desktop for desktop devices.
revenue — revenue from the order.
service_name — name of the ticket operator.
tickets_count — number of tickets purchased.
total — total order amount. 
The second dataset, final_tickets_events_df, contains information about events, including the city and region of the event, as well as information about the event venue:
event_id — unique event identifier.
event_name — event name. Equivalent to the event_name_code field from the original database.
event_type_description — description of the event type.
event_type_main — main type of the event: theatrical performance, concert, and so on.
organizers — event organizers.
region_name — region name.
city_name — city name.
venue_id — unique venue identifier.
venue_name — venue name.
venue_address — venue address.
The third dataset, final_tickets_tenge_df.csv, contains information about the tenge to Russian ruble exchange rate for the year 2024:
nominal — nominal value (100 tenge).
data — date.
curs — tenge to ruble exchange rate.
cdx — currency designation (kzt).*

## Project Structure

1. Formatting the Jupyter Notebook.
2. Loading data and getting to know it.
3. Data preprocessing and preparation for analysis.
4. Exploratory data analysis:
- Analysis of order distribution by segments and their seasonal changes,
- Autumn user activity,
- Popular events and partners.
5. Statistical data analysis: We will test two hypotheses suggesting higher activity of mobile device users:
- The average number of orders per mobile app user is higher compared to desktop users,
- The average time between orders for mobile app users is higher compared to desktop users.
6. General conclusion and recommendations.

## Loading data and getting to know it

In [ ]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy.stats import mannwhitneyu

In [ ]:
# Exporting ticket order data
df_orders = pd.read_csv('/datasets/final_tickets_orders_df.csv')
# Exporting data with information about events
df_events = pd.read_csv('/datasets/final_tickets_events_df.csv')
# Exporting data with information on the exchange rate of the tenge to the Russian ruble for 2024
df_tenge = pd.read_csv('/datasets/final_tickets_tenge_df.csv')

In [ ]:
# Let's print the first 5 rows about the ticket orders dataframe
df_orders.head()

In [ ]:
# Display information about the dataframe on ticket orders
df_orders.info()

In [ ]:
# Let's display the first 5 rows of the dataframe with information about events
df_events.head()

In [ ]:
# Let's display information about the dataframe with information on events
df_events.info()

In [ ]:
# Let's display information about the dataframe on the tenge to Russian ruble exchange rate for 2024.
df_tenge.info()

***Intermediate conclusion:  
The data has been loaded correctly.  
The orders table df_orders contains the necessary parameters and only a small number of missing values, only in the days_since_prev field.  
There are no missing values in the events data df_events.  
A dataset with tenge exchange rates will be needed to convert revenue to a single currency, rubles.  
After the initial analysis, the following steps for data processing have been identified:   
Convert created_dt_msk and created_ts_msk to datetime format.  
Convert categorical fields (device_type_canonical, service_name, cinema_circuit, region_name, etc.) to category type.  
Check numerical features (total, revenue, tickets_count, days_since_prev) for correct ranges.  
There are about 22 thousand missing values in days_since_prev, which is logical because there are no previous orders, so we will leave the missing values and use only the rows with filled values when analyzing intervals.  
-Check for duplicates, remove duplicates if found.
-Check for missing values in the other tables — make sure there are no critically important empty values.
-Check for possible outliers in total and revenue.
-Create additional columns: day of the week, month, season, week of the year.
-Normalize categories, if necessary bring all categories to a uniform format.***

## Data preprocessing and preparation for analysis

In [ ]:
# Missing values by columns
print("Missing values in df_orders:")
display(df_orders.isna().sum())

print("nMissing values in df_events:")
display(df_events.isna().sum())

print("nMissing values in df_tenge:")
display(df_tenge.isna().sum())


In [ ]:
# let's see what categories exist in the categorical columns for df_orders
cat_cols_orders = ['currency_code', 'device_type_canonical', 'cinema_circuit', 'service_name']

for col in cat_cols_orders:
    print(f'nCategories in column "{col}":')
    print(df_orders[col].unique())

In [ ]:
# let's see what categories exist in the categorical columns for df_events
cat_cols_events = ['event_type_main', 'event_type_description',
'region_name', 'city_name', 'venue_name', 'organizers']

for col in cat_cols_events:
    print(f'nCategories in column "{col}":')
    print(df_events[col].unique())

In [ ]:
# Let's rename the category 'no' to a more appropriate one - 'Offline'
df_orders['cinema_circuit'] = df_orders['cinema_circuit'].replace({'нет': 'Без сети'})

***After analyzing the categorical columns, it can be noted that there are no critical issues with the quality of the categories in the data.  
Categories that do not require normalization:  
currency_code: only rub and kzt.  
device_type_canonical: mobile and desktop.  
event_type_main: 7 main types, without duplicates.  
event_type_description: diverse but correct subtypes without spelling duplicates.  
organizers: code values like "№XXXX", normalization is not needed.  
venue_name: thousands of unique values.  
organizers: formal identifiers.  
cinema_circuit contains:  
'нет', 'Другое', 'Киномакс', 'КиноСити', 'Москино', 'ЦентрФильм'.  
Category "нет" — is effectively the absence of a chain, can be converted to None or No chain.  
Conclusion:  
-There are no serious issues in the data that require mandatory normalization.  
-Most categorical features are structured correctly and do not contain spelling duplicates. ***

In [ ]:
# Let's split the data by currency using copy()
orders_rub = df_orders[df_orders['currency_code'] == 'rub'].copy()
orders_kzt = df_orders[df_orders['currency_code'] == 'kzt'].copy()


In [ ]:
# Revenue statistics in rubles and tenge
display(orders_rub['revenue'].describe())
display(orders_kzt['revenue'].describe())
# Ticket Quantity Statistics
display(df_orders['tickets_count'].describe())

In [ ]:
# Histogram of revenue distribution in rubles
plt.figure(figsize=(12,5))
plt.hist(orders_rub['revenue'], bins=50)
plt.title("Revenue distribution (rub)")
plt.xlabel("Revenue")
plt.ylabel("Quantity")
plt.show()
# Histogram of revenue distribution in tenge
plt.figure(figsize=(12,5))
plt.hist(orders_kzt['revenue'], bins=50)
plt.title("Revenue distribution (kzt)")
plt.xlabel("Revenue")
plt.ylabel("Quantity")
plt.show()
# Histogram of distribution by number of tickets
plt.figure(figsize=(12,5))
plt.hist(df_orders['tickets_count'], bins=50)
plt.title("Distribution of 'tickets_count'")
plt.xlabel("Number of tickets")
plt.ylabel("Quantity")
plt.show()


In [ ]:
# Let's look at the boxplot for the concept of the number of outliers for rubles and tenge
plt.figure(figsize=(10,4))
plt.boxplot(orders_rub['revenue'])
plt.title("Revenue Boxplot (rub)")
plt.show()

plt.figure(figsize=(10,4))
plt.boxplot(orders_kzt['revenue'])
plt.title("Revenue Boxplot (kzt)")
plt.show()

plt.figure(figsize=(10,4))
plt.boxplot(df_orders['tickets_count'])
plt.title("Boxplot Number of Tickets")
plt.show()

In [ ]:
# Outlier search by 99th percentile
p99_rub = orders_rub['revenue'].quantile(0.99)
p99_kzt = orders_kzt['revenue'].quantile(0.99)

display("99th percentile of revenue (rub):", p99_rub)
display("99th percentile of revenue (kzt):", p99_kzt)

In [ ]:
# Let's filter the data by the 99th percentile
orders_rub_filtered = orders_rub[orders_rub['revenue'] <= p99_rub]
orders_kzt_filtered = orders_kzt[orders_kzt['revenue'] <= p99_kzt]

In [ ]:
# Save the filtering results in df_orders
df_orders = df_orders[((df_orders['currency_code'] == 'rub') & (df_orders['revenue'] <= p99_rub)) |
          ((df_orders['currency_code'] == 'kzt') & (df_orders['revenue'] <= p99_kzt))].copy()

***After splitting the data by currencies, the distributions of revenue and tickets_count values were studied.  
The revenue distributions in RUB and KZT turned out to be heavily right-skewed, with most orders being small and large orders being rare.  
Outliers that significantly exceed the main range of values are visible on the boxplot and histograms.  
For accurate analysis, the 99th percentiles were calculated:  
values above the 99th percentile are considered outliers.  
Next, the data were filtered by revenue for rubles and tenge.  
The number of tickets (tickets_count) also has rare large values, but the outliers are moderate and not numerous.***

In [ ]:
# Checking for explicit duplicates
df_orders.duplicated().sum()
df_events.duplicated().sum()
display("Explicit duplicates in orders:", df_orders.duplicated().sum())
display("Explicit duplicates in events:", df_events.duplicated().sum())

In [ ]:
# Check for implicit duplicates. Create a list of duplicates excluding order_id
cols_no_order = df_orders.columns.drop('order_id')
# Check the sum
df_orders.duplicated(subset=cols_no_order).sum()

In [ ]:
# Let's display examples
df_orders[df_orders.duplicated(subset=cols_no_order, keep=False)].head(30)

In [ ]:
# Let's remove implicit duplicates leaving only one instance each.
df_orders = df_orders.drop_duplicates(subset=cols_no_order, keep='first').copy()
# Let's check
df_orders.duplicated(subset=cols_no_order).sum()

***No explicit duplicates were found.  
Implicit duplicates were detected: several rows have identical values in almost all columns, except only for the order_id field and sometimes user_id.  
These are technical duplicates of the same transaction, duplicated in the partners' export. Therefore, we will remove such duplicates across all columns except order_id, keeping one value.***

In [ ]:
# Let's try to find implicit duplicates, list of columns of interest
columns_to_check = ['user_id', 'created_ts_msk', 'cinema_circuit', 'service_name']
# Output unique values for each column
for column in columns_to_check:
    print(f'nUnique values in column "{column}":')
    print(df_orders[column].unique())

In [ ]:
# Let's convert the dates to datetime
df_orders['created_dt_msk'] = pd.to_datetime(df_orders['created_dt_msk'], errors='coerce')
df_orders['created_ts_msk'] = pd.to_datetime(df_orders['created_ts_msk'], errors='coerce')

In [ ]:
# Let's reduce the dimensionality of numerical columns
df_orders['age_limit'] = df_orders['age_limit'].astype('int8')
df_orders['tickets_count'] = df_orders['tickets_count'].astype('int8')
df_orders['revenue'] = df_orders['revenue'].astype('float32')
df_orders['total'] = df_orders['total'].astype('float32')
df_orders['order_id'] = df_orders['order_id'].astype('int32')
df_events['event_id'] = df_events['event_id'].astype('int32')
df_events['venue_id'] = df_events['venue_id'].astype('int32')
df_events['city_id'] = df_events['city_id'].astype('int32')

In [ ]:
# Convert categorical columns to category
df_orders['currency_code'] = df_orders['currency_code'].astype('category')
df_orders['device_type_canonical'] = df_orders['device_type_canonical'].astype('category')
df_events['event_type_main'] = df_events['event_type_main'].astype('category')
df_events['event_type_description'] = df_events['event_type_description'].astype('category')

In [ ]:
df_tenge.columns

In [ ]:
# Let's combine the dataframes df_orders and df_events
df = df_orders.merge(df_events, on='event_id', how='left')
# Convert the exchange rate to rubles, convert the date to a proper format, and keep the necessary columns
df_tenge['data'] = pd.to_datetime(df_tenge['data'])
df_tenge = df_tenge.rename(columns={'data': 'created_dt_msk'})
# Merge with the orders
df = df.merge(df_tenge[['created_dt_msk', 'curs']], on='created_dt_msk', how='left')

In [ ]:
# Pulling the exchange rate by date
rate_map = df_tenge['curs']
df_orders['tenge_rate'] = df_orders['created_dt_msk'].map(rate_map)
# Initializing a new column with a default value
rate_map = df_tenge['curs']
df_orders['tenge_rate'] = df_orders['created_dt_msk'].map(rate_map)
# Creating the revenue_rub column
df_orders['revenue_rub'] = df_orders['revenue']
# Creating a mask for rows
mask_kzt = df_orders['currency_code'] == 'kzt'
# Performing a vectorized operation only for the required rows
df_orders.loc[mask_kzt, 'revenue_rub'] = (df_orders.loc[mask_kzt, 'revenue'] * df_orders.loc[mask_kzt, 'tenge_rate'] / 100)

In [ ]:
# let's calculate the revenue from the sale of one ticket to the event
df['one_ticket_revenue_rub'] = df_orders['revenue_rub'] / df_orders['tickets_count']

In [ ]:
# Highlight the order placement month
df['month'] = df['created_dt_msk'].dt.month

In [ ]:
# Creating a column season
def get_season(month):
    if month in [6, 7, 8]:
        return 'summer'
    elif month in [9, 10, 11]:
        return 'autumn'
    elif month in [12, 1, 2]:
        return 'winter'
    else:
        return 'spring'
df['season'] = df['month'].apply(get_season)

In [ ]:
# Check result
df.head()

***Intermediate conclusion:  
We preprocessed the data, found no explicit or implicit duplicates, converted the time columns to datetime, reduced the dimensionality of numerical columns by converting them to int8, int32, and float32, and transformed categorical columns.  
As part of preprocessing, data from two sets—orders and events—were merged by the event_id key. Since revenue was indicated in two currencies (Russian rubles and Kazakh tenge), we converted all currencies to rubles. For this, we used a dataset of the tenge exchange rate, at a rate of 100 tenge per ruble.  
New columns were added to the table:  
revenue_rub — revenue converted to Russian rubles;  
one_ticket_revenue_rub — average revenue per one sold ticket;  
month — month number of the order;  
season — season of the order ('summer', 'autumn', 'winter', 'spring').***

## Exploratory Data Analysis

**7.1. Analysis of Order Distribution by Segments and Their Seasonal Changes**

In [ ]:
# Number of orders by month
monthly_orders = df.groupby('month')['order_id'].count().reset_index(name='orders_count')
# Let's build a bar chart
plt.figure(figsize=(10,5))
sns.barplot(data=monthly_orders, x='month', y='orders_count')
plt.title('Number of orders by month (June-October 2024)')
plt.xlabel('Month')
plt.ylabel('Number of orders')
plt.show()

In [ ]:
# Distribution of orders by segments in summer and autumn, let's highlight only summer and autumn
df_seasons = df[df['season'].isin(['summer', 'autumn'])]


In [ ]:
# Distribution by event type, counting the number of orders by season and event type
season_event = df_seasons.groupby(['season','event_type_main'], as_index=False)['order_id'].count()
# Counting the total number of orders by season
season_totals = df_seasons.groupby('season', as_index=False)['order_id'].count()
season_totals = season_totals.rename(columns={'order_id': 'season_total'})
# Merging
season_event = season_event.merge(season_totals, on='season', how='left')
# Calculating the share
season_event['share'] = season_event['order_id'] / season_event['season_total']
# Visualization
plt.figure(figsize=(12,6))
sns.barplot(data=season_event, x='event_type_main', y='share', hue='season')
plt.title('Share of orders by event type in summer and autumn')
plt.xlabel('Event type')
plt.ylabel('Share of orders')
plt.legend(title='Season')
plt.show()

In [ ]:
# Distribution by device type, counting the number of orders
season_device = df_seasons.groupby(['season','device_type_canonical'], as_index=False)['order_id'].count()
# Sum by season
device_totals = df_seasons.groupby('season', as_index=False)['order_id'].count()
device_totals = device_totals.rename(columns={'order_id': 'season_total'})
# Merge and calculate the share
season_device = season_device.merge(device_totals, on='season', how='left')
season_device['share'] = season_device['order_id'] / season_device['season_total']
# Visualization
plt.figure(figsize=(8,5))
sns.barplot(data=season_device, x='device_type_canonical', y='share', hue='season')
plt.title('Share of orders by device type in summer and autumn')
plt.xlabel('Device type')
plt.ylabel('Share of orders')
plt.show()

In [ ]:
# Distribution by age category, counting the number of orders
season_age = df_seasons.groupby(['season','age_limit'], as_index=False)['order_id'].count()
# Total by season
age_totals = df_seasons.groupby('season', as_index=False)['order_id'].count()
age_totals = age_totals.rename(columns={'order_id': 'season_total'})
# Merge and calculate share
season_age = season_age.merge(age_totals, on='season', how='left')
season_age['share'] = season_age['order_id'] / season_age['season_total']
# Visualization
plt.figure(figsize=(10,5))
sns.barplot(data=season_age, x='age_limit', y='share', hue='season')
plt.title('Share of orders by age category in summer and autumn')
plt.xlabel('Age limit')
plt.ylabel('Share of orders')
plt.show()


In [ ]:
# Average cost of a single ticket by event type, calculating the average ticket price
avg_ticket_price = df_seasons.groupby(['season','event_type_main'], as_index=False)['one_ticket_revenue_rub'].mean()
avg_ticket_price = avg_ticket_price.rename(columns={'one_ticket_revenue_rub': 'avg_ticket_rub'})
avg_ticket_price.head()
# Calculating the relative change from autumn to summer
pivot_price = avg_ticket_price.pivot(index='event_type_main', columns='season', values='avg_ticket_rub').reset_index()
# Relative change in %
pivot_price['relative_change_%'] = (pivot_price['autumn'] - pivot_price['summer']) / pivot_price['summer'] * 100
pivot_price

In [ ]:
# Visualization of the average ticket price
pivot_price_plot = pivot_price.set_index('event_type_main')[['summer','autumn']]

pivot_price_plot.plot(kind='bar', figsize=(12,6))
plt.title('Average check per ticket by event type in summer and autumn')
plt.ylabel('Revenue per ticket (RUB)')
plt.xlabel('Event type')
plt.xticks(rotation=45)
plt.show()

***Conclusion:
The distribution of orders by event types is generally stable, with a slight trend that more people attend concerts in the summer and theater in the autumn. The biggest difference is in the sports category, which is more popular in the autumn, possibly due to a break in the sports leagues during the summer.
The distribution of orders by device type hardly changes across seasons.
The distribution of orders by age restriction also shows no significant difference; some restrictions are slightly higher in summer, others in autumn.
Dynamics of average cost:
There is a slight predominance of average ticket prices in summer compared to autumn. The category leaders by the summer-autumn difference are theater, New Year’s shows, and concerts (with a 10% to 17% higher average in summer than in autumn).
Conclusion on average check:
For all event types, there is no significant shift in average revenue between the summer and autumn seasons.***

**7.2. Autumn user activity**

In [ ]:
# Filter only the autumn months of 2024
df_autumn = df[df['season'] == 'autumn'].copy()
# Create a column with the date
df_autumn['order_date'] = df_autumn['created_dt_msk'].dt.date

In [ ]:
# Let's create a pivot table by days
daily_stats = df_autumn.groupby('order_date').agg(
orders_count=('order_id','count'),
dau=('user_id','nunique'),
avg_ticket_price=('one_ticket_revenue_rub','mean')).reset_index()
# Orders per user
daily_stats['orders_per_user'] = daily_stats['orders_count'] / daily_stats['dau']
daily_stats.head()

In [ ]:
# Dynamics of the number of orders
plt.figure(figsize=(12, 5))
plt.plot(daily_stats['order_date'], daily_stats['orders_count'])
plt.title('Number of orders by day (Autumn 2024)')
plt.xlabel('Date')
plt.ylabel('Number of orders')
plt.xticks(rotation=45)
plt.grid(True)
plt.show()

In [ ]:
# DAU Dynamics
plt.figure(figsize=(12, 5))
plt.plot(daily_stats['order_date'], daily_stats['dau'])
plt.title('DAU — number of active users by day (autumn 2024)')
plt.xlabel('Date')
plt.ylabel('Unique users')
plt.xticks(rotation=45)
plt.grid(True)
plt.show()

In [ ]:
# Dynamics of the average number of orders per user
plt.figure(figsize=(12,5))
plt.plot(daily_stats['order_date'], daily_stats['orders_per_user'])
plt.title('Average number of orders per user by day (Autumn 2024)')
plt.xlabel('Date')
plt.ylabel('Orders per user')
plt.xticks(rotation=45)
plt.grid(True)
plt.show()

In [ ]:
# Dynamics of the average price of a single ticket
plt.figure(figsize=(12,5))
plt.plot(daily_stats['order_date'], daily_stats['avg_ticket_price'])
plt.title('Average price of a single ticket by day (autumn 2024)')
plt.xlabel('Date')
plt.ylabel('Ticket price, rubles')
plt.xticks(rotation=45)
plt.grid(True)
plt.show()

In [ ]:
# Add the day of the week (0 - Monday, 6 - Sunday)
df_autumn['weekday'] = df_autumn['created_dt_msk'].dt.dayofweek
# Create a dictionary for displaying the days of the week
weekday_map = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri', 5: 'Sat', 6: 'Sun'}
df_autumn['weekday_name'] = df_autumn['weekday'].map(weekday_map)
# Group by each day of the week
weekday_stats = df_autumn.groupby('weekday_name').agg(
    orders_count=('order_id','count'),
    dau=('user_id','nunique'),
    avg_ticket_price=('one_ticket_revenue_rub','mean')).reset_index()
# Calculate the average number of orders per user
weekday_stats['orders_per_user'] = weekday_stats['orders_count'] / weekday_stats['dau']
weekday_stats.head(7)


In [ ]:
# Setting the order of the days of the week
weekday_order = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
weekday_stats = weekday_stats.set_index('weekday_name').reindex(weekday_order)

In [ ]:
# Orders count plot
plt.figure(figsize=(10,5))
weekday_stats['orders_count'].plot(kind='bar')
plt.title('Number of orders by day of the week')
plt.ylabel('orders_count')
plt.xlabel('Day of the week')
plt.show()

In [ ]:
# Unique Users Chart (DAU)
plt.figure(figsize=(10,5))
weekday_stats['dau'].plot(kind='bar')
plt.title('Number of Unique Users by Day of the Week')
plt.ylabel('dau')
plt.xlabel('Day of the Week')
plt.show()


In [ ]:
# Average number of orders per user chart
plt.figure(figsize=(10,5))
weekday_stats['orders_per_user'].plot(kind='bar')
plt.title('Average number of orders per user by day of the week')
plt.ylabel('orders_per_user')
plt.xlabel('Day of the week')
plt.show()

In [ ]:
# Average ticket price chart by day
plt.figure(figsize=(10,5))
weekday_stats['avg_ticket_price'].plot(kind='bar')
plt.title('Average price of a single ticket by day')
plt.ylabel('orders_per_user')
plt.xlabel('Day of the week')
plt.show()

***Conclusion:  
The dynamics of the number of orders per day are unstable, ranging from around 1,000 per day to 7,000 per day.  
The dynamics of the number of active users in autumn 2024 are positive, with the number gradually increasing from less than 600 and growing to 1,300 users per day by the end of October.  
The average number of orders per day has an average value of 2 to 3, but there are periods of spikes up to 6–7 orders, and the graph shows that these periods occur at the beginning of the month. It can be assumed that the average number of orders increases at the start of the month.  
The average ticket cost ranged from 180 to 200 rubles and has relatively stable dynamics. 
Depending on the day of the week, the dynamics are as follows: 
In terms of the number of orders, the most active days are Tuesday and Thursday, with the fewest orders on Monday, Saturday, and Sunday. 
The number of unique DAU users is highest on Thursday and lowest on Sunday, ranging from 4394 to 4970; it can be said that there is no significant difference. 
The average number of orders is highest on Tuesday (about 6.5) and lowest on Saturday and Sunday, suggesting that users place fewer orders on weekends. 
The average cost of a single ticket ranges from 186 to 191 rubles; it can be said that this metric is the most stable and does not depend on the day of the week.***

**7.3. Popular events and partners**

In [ ]:
# Analysis by regions. Let's count the number of unique events and total number of orders
region_stats = df_autumn.groupby('region_name', as_index=False).agg(
unique_events=('event_id','nunique'),
orders_count=('order_id','count'))
# Let's calculate the shares
region_stats['events_share'] = region_stats['unique_events'] / region_stats['unique_events'].sum()
region_stats['orders_share'] = region_stats['orders_count'] / region_stats['orders_count'].sum()
# Sort by event diversity (unique_events)
region_stats = region_stats.sort_values(by='unique_events', ascending=False)
region_stats.head(10)

In [ ]:
# Visualization of the top 10 regions by the number of unique events
plt.figure(figsize=(12,6))
sns.barplot(data=region_stats.head(10), x='region_name', y='unique_events')
plt.title('Top 10 regions by the number of unique events')
plt.xlabel('Region')
plt.ylabel('Number of unique events')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Let's create the column revenue_rub in df_autumn
df_autumn['revenue_rub'] = df_autumn['revenue']
# Analysis by ticket partners. Let's calculate the number of unique events, the number of processed orders, and total revenue.
partner_stats = df_autumn.groupby('service_name', as_index=False).agg(
    unique_events=('event_id','nunique'),
    orders_count=('order_id','count'),
    revenue_rub=('revenue_rub','sum'))
# Convert revenue to millions of rubles and round
partner_stats['revenue_rub_mln'] = (partner_stats['revenue_rub'] / 1000000).round(2)
# Let's calculate the shares
partner_stats['events_share'] = partner_stats['unique_events'] / partner_stats['unique_events'].sum()
partner_stats['orders_share'] = partner_stats['orders_count'] / partner_stats['orders_count'].sum()
partner_stats['revenue_share'] = partner_stats['revenue_rub'] / partner_stats['revenue_rub'].sum()
# Sort by the number of processed orders
partner_stats = partner_stats.sort_values(by='orders_count', ascending=False)
# View top 10 partners
partner_stats[['service_name','unique_events','orders_count','revenue_rub_mln']].head(10)

In [ ]:
# Visualization of the top 10 partners by the number of processed orders
plt.figure(figsize=(12,6))
sns.barplot(data=partner_stats.head(10), x='service_name', y='orders_count')
plt.title('Top 10 partners by number of orders')
plt.xlabel('Partner')
plt.ylabel('Number of processed orders')
plt.xticks(rotation=45)
plt.show()

***Conclusion:  
The top regions by the number of unique events were: Kamenevsky Region, Severoyarskaya Oblast, Shirokovskaya Oblast, Svetopolyansky District. The number of unique events over 2 months of autumn in these regions ranged from 700 to almost 4000 for the clear leader – Kamenevsky District, accounting for 25% of the total number of unique events across all regions.  
Top partners among ticket operators by number of orders were 'Tickets Without Problems', 'Catch a Ticket!', and 'Tickets in Hand'. The number of processed orders by 'Tickets Without Problems' amounted to 19% of all partners.***

##  Statistical Data Analysis

**8.1 Formulation of the null and alternative hypotheses.  
Hypothesis 1: number of orders per user:  
H0 (null hypothesis): the average number of orders per user of the mobile application is not greater than that of desktop users.  
H1 (alternative): the average number of orders per user of the mobile application is higher than that of desktop users.**

In [ ]:
# Let's prepare the data for hypotheses. We count the number of orders per user for each device category
orders_per_user = df_autumn.groupby(['user_id','device_type_canonical'])['order_id'].count().reset_index()
# We split into mobile and desktop
mobile_orders = orders_per_user[orders_per_user['device_type_canonical'] == 'mobile']['order_id']
desktop_orders = orders_per_user[orders_per_user['device_type_canonical'] == 'desktop']['order_id']

In [ ]:
# Let's check the normality of the distribution using a histogram
plt.figure(figsize=(20,15))
sns.histplot(mobile_orders, bins=30, color='blue', label='Mobile', stat='density', common_norm=False)
sns.histplot(desktop_orders, bins=30, color='orange', label='Desktop', alpha=0.6, stat='density', common_norm=False)
plt.title('Distribution of number of orders per user (normalized)')
plt.xlabel('Number of orders')
plt.ylabel('Density of distribution')
plt.legend()
plt.show()

**Most users have a very small number of orders from 0 to 5, while on the X-axis there are rare values up to several thousand orders. These are signs of a long right tail and outliers because the distribution is heavily skewed to the right and has extreme values compared to the median. The Mann–Whitney test does not require normality and is suitable for assessing shifts in distributions, not just means, which is why we chose it as a non-parametric test in case of strong asymmetry or outliers (there are no requirements that there should be no heavy tails, no outliers, or that the sample mean must have a normal distribution). It helps to check whether distributions differ between two samples using ranks rather than absolute values.**

In [ ]:
# Mann-Whitney statistical test
alpha = 0.05
stat, p_value = mannwhitneyu(mobile_orders, desktop_orders, alternative='greater')
display(f"p-value: {p_value}")
if p_value < alpha:
    display("we reject the null hypothesis H0, the average number of orders per mobile app user is higher compared to desktop users.")
else:
    display("there is no statistically significant difference, we accept the null hypothesis H0")

**8.2 Hypothesis Formulation:  
Hypothesis 2: Average time between orders:  
H0: The average time between orders of mobile users is equal to the average for stationary users.  
H1: The average time between orders of mobile users is greater than that of stationary users.**

In [ ]:
# Filter only records with a non-zero interval
df_intervals = df_autumn.dropna(subset=['days_since_prev'])

# Split into mobile and desktop devices
mobile_intervals = df_intervals[df_intervals['device_type_canonical'] == 'mobile']['days_since_prev']
desktop_intervals = df_intervals[df_intervals['device_type_canonical'] == 'desktop']['days_since_prev']

In [ ]:
# Let's check the samples for missing values
print(mobile_intervals.isna().sum())
print(desktop_intervals.isna().sum())

In [ ]:
# Let's remove these gaps
mobile_intervals = mobile_intervals.dropna()
desktop_intervals = desktop_intervals.dropna()

In [ ]:
# Let's check the statistics. how skewed the values are
mobile_intervals.describe()
desktop_intervals.describe()

**Similarly to the first hypothesis, the Mann-Whitney test will help determine whether the distributions between the two samples differ, using ranks rather than absolute values. From the description of the statistics, it is clear that with a mean of around 3 and a standard deviation of about 12, the maximum is nearly 146, from which it can be concluded that the distribution is heavily right-skewed, which are clear signs of a heavy tail. The Mann-Whitney test has no requirements regarding heavy tails, outliers, or the sample mean needing to have a normal distribution, so we choose it again.**

In [ ]:
alpha = 0.05
stat2, p_value2 = mannwhitneyu(mobile_intervals, desktop_intervals, alternative='greater')
print(f"p-value: {p_value2}")
if p_value < alpha:
    display("The average time between orders of mobile app users is higher compared to desktop users, the null hypothesis H0 is rejected, hypothesis H1 is confirmed.")
else:
    display("The average time between orders of mobile app users is lower compared to desktop users, the null hypothesis is not rejected")

## General conclusion and recommendations.

***Data Description:
The data was loaded correctly.***

***Missing Values
-In df_orders, a small number of missing values are present only in the days_since_prev column. This is expected, as users making their first purchase do not have a previous order.
-df_events contains no missing values.
-The currency rates table has no missing values.***

***Preprocessing Steps:
-Converted created_dt_msk and created_ts_msk to datetime format.
-Converted categorical columns (device_type_canonical, service_name, cinema_circuit, region_name, etc.) to category type.
-Checked numerical features (total, revenue, tickets_count, days_since_prev) for correctness.
-Created additional features: day of the week, month, season, week of the year.
-In cinema_circuit, logically replaced “нет” with “No Network”.
-Converted revenue from KZT to RUB.***

***Distribution and Outlier Analysis:
-Revenue (revenue) distributions in RUB and KZT are highly right-skewed: many small orders and few large ones.
-Outliers are visible on histograms and boxplots.
-The 99th percentile was used to trim extremely high values for both RUB and KZT.
-tickets_count has a few rare high values, distribution is moderately skewed.***

***Duplicate Analysis:
-No exact duplicates found.
Implicit duplicates were detected: rows fully identical across key fields except for order_id and user_id.***

***Data Preparation
Data was cleaned, merged, and enriched with the following features for analysis:
-season — season of purchase
-revenue_rub — revenue in RUB
-one_ticket_revenue_rub — revenue per ticket
-Aggregates computed by user, date, day of the week, device type, and region.***

***Main Analysis Results
Event Popularity and Seasonal Dynamics:
-In summer, users mostly attend concerts; in autumn, theater and sports events.
-The largest seasonal difference is in sports events, which attract more viewers in autumn.***

***Device Type and Age Restrictions
-Distribution of orders by device type is stable across seasons.
-Age restriction distribution is stable; no seasonal skew observed.***

***Average Ticket Price
-Slightly higher in summer than autumn.
-Largest increases in summer are in theater, “Yolki” (New Year shows), and concerts (10–17% higher than in autumn).***

***User Activity in Autumn
-Daily order counts fluctuate between ~1,000 and 7,000.
-Active users gradually increased from under 600 in early autumn to 1,300 by late October.
-Average orders per day range from 2 to 3, with spikes of 6–7 orders at the beginning of each month.
-Average ticket price ranged from 180 to 200 RUB, relatively stable.***

***By weekday:
-Highest order counts on Tuesday and Thursday; lowest on Monday, Saturday, and Sunday.
-DAU peaks on Thursday, lowest on Sunday (4394–4970 users).
-Average orders per user highest on Tuesday (~6.5), lowest on weekends.
-Ticket price is stable (186–191 RUB) and not dependent on the day of the week.***

***Regional and Partner Activity
-Top regions by unique events: Kamenevsky, Severoyarskaya, Shirokovskaya, and Svetopolyansky.
-Kamenevsky leads with ~4,000 events (25% of total).
-Top partners by order volume: “Bilety bez problem”, “Lovi Bilet!”, “Bilety v Ruki”.
-“Bilety bez problem” accounts for 19% of all orders.***

***Hypothesis Testing Results
Hypothesis 1 — Average Orders per User:
-Mobile app users have a higher average number of orders than desktop users.
-Null hypothesis H0 rejected; alternative H1 confirmed.
Conclusion: Mobile users are more active in placing orders, likely due to convenience and accessibility of the app.***

***Hypothesis 2 — Average Time Between Orders
-Mobile users have longer average intervals between orders compared to desktop users.
-H0 rejected; H1 confirmed.
Conclusion: Despite higher frequency of orders, mobile users place them at longer intervals, possibly due to seasonality or app usage patterns.***

***Recommendations
Seasonal Focus:
-Increase marketing for theater and sports events in autumn; for concerts in summer.***

***Partner Optimization:
-Collaboration with top partners (“Bilety bez problem”, “Lovi Bilet!”) generates significant revenue.
-Evaluate less active partners to expand reach.***

***Regional Strategy:
-Focus on Kamenevsky, Severoyarskaya, and Shirokovskaya regions with most events.
-Implement campaigns and promotions in less active regions to boost sales.***

***User Activity & Marketing
-Weekdays are key for promotions and marketing campaigns.
-Segment users by device type and provide targeted offers for mobile users.***